# v3_dano: segmentador de la zona dañada del huevo

**Qué añade.** `v2` detecta cada huevo y dice si está rajado. `v3_dano` es un segundo modelo, pequeño, que recibe el **recorte de cada huevo** y devuelve dos máscaras: la **silueta del huevo** y la **zona dañada**. Con ellas la app puede pintar dónde está la grieta y calcular la **gravedad** (% de la cáscara dañada).

**Por qué un segundo modelo y no un YOLOv8-seg.** Solo hay zona dañada anotada en ~360 de los 2110 huevos Crack de train. En un único modelo de segmentación, los Crack sin anotar le enseñarían "aquí no hay daño". Además, al recortar el huevo del frame completo, las grietas finas se ven con más resolución.

**Datos.** Las anotaciones están en el repo (`dano/anotaciones/`): zonas marcadas sobre una rejilla 10×10 por huevo y suavizadas. La silueta del huevo sale de SAM 2.1 usando la caja de la etiqueta. Los negativos son huevos Intact (sin daño).

**Pasos** (Colab con GPU, de arriba abajo):
1. Setup: Drive, repo, dataset.
2. Dataset de recortes (SAM + anotaciones). Se guarda en Drive y se reutiliza.
3. Revisión visual de las máscaras.
4. Entrenamiento del run `v3_dano` (no sobrescribe runs existentes).
5. Exportación a `.tflite` y evaluación de la tubería completa (detector `v2` → recorte → segmentador).

## 1. Setup
Monta Drive, clona el repo (código y anotaciones de `dano/`), descomprime el dataset en `/content/eggs_v2` si no está y prepara la carpeta de trabajo. Comprueba que hay GPU.

In [ ]:
%pip install -q ultralytics

import os, subprocess, zipfile, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/eggs_v2')
DATA_DIR = Path('/content/eggs_v2')
REPO = Path('/content/eggs-detector')
WORK = Path('/content/dano_work')                  # aquí corren los scripts (rutas relativas data/, masks/)
SEG_DIR = DRIVE_DIR / 'dano' / 'seg'               # dataset de recortes (se guarda en Drive)
RUNS_DIR = DRIVE_DIR / 'runs'
EXPORTS_DIR = DRIVE_DIR / 'exports'
RUN = 'v3_dano'
DET = EXPORTS_DIR / 'v2' / 'eggs_v2_fp32.tflite'   # detector entregado, para evaluar la tubería completa

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/dportilla219/eggs-detector.git', str(REPO)], check=True)

if not (DATA_DIR / 'train').exists():
    for n in ['eggs_v2_parte1_train.zip', 'eggs_v2_parte2_train.zip',
              'eggs_v2_parte3_train.zip', 'eggs_v2_parte4_valid_test.zip']:
        with zipfile.ZipFile(DRIVE_DIR / n) as zf:
            zf.extractall(DATA_DIR)
        print('OK', n)

WORK.mkdir(exist_ok=True)
if not (WORK / 'data').exists():
    (WORK / 'data').symlink_to(DATA_DIR)
for f in (REPO / 'dano' / 'anotaciones').glob('*.txt'):
    shutil.copy(f, WORK)
os.chdir(WORK)

os.environ['SEG_DIR'] = str(SEG_DIR)
os.environ['RUNS_DIR'] = str(RUNS_DIR)
os.environ['PYTHONPATH'] = str(REPO / 'dano')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
assert DET.exists(), f'Falta {DET}'

import tensorflow as tf
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 2. Dataset de recortes
Para cada huevo anotado (y para los negativos Intact) calcula la silueta con SAM 2.1 en GPU y convierte las celdas marcadas en la máscara de zona dañada. El resultado va a `MyDrive/eggs_v2/dano/seg`. Si ya existe, se salta.

In [ ]:
D = REPO / 'dano'
if (SEG_DIR / 'meta.json').exists():
    print('Dataset ya generado en', SEG_DIR)
else:
    for split, lst in [('train', 'items_train.txt'), ('train', 'items_neg.txt'),
                       ('valid', 'items_valid.txt'), ('valid', 'items_neg_valid.txt'),
                       ('test', 'items_test.txt'), ('test', 'items_neg_test.txt')]:
        !python {D}/eggmask.py {split} 999999 {lst} 2>&1 | grep -v Warn | tail -1
    SEG_DIR.mkdir(parents=True, exist_ok=True)
    !python {D}/build.py
for s in ['train', 'valid', 'test']:
    fs = list((SEG_DIR / s).glob('*.npz'))
    print(s, len(fs), 'recortes |', sum(f.name.endswith('_p.npz') for f in fs), 'con daño')

## 3. Revisión visual
Doce recortes de train al azar: a la izquierda el original y a la derecha la silueta (verde) y la zona dañada anotada (magenta).

In [ ]:
from IPython.display import Image, display
!python {REPO}/dano/segview.py train 1 /content/revision.jpg
display(Image('/content/revision.jpg', width=900))

## 4. Entrenamiento `v3_dano`
MobileNetV2 (α 0.5, preentrenado en ImageNet) + decodificador U-Net. Entrada `192×192×3` RGB 0–1; salida `192×192×2` con sigmoide (canal 0 = huevo, canal 1 = daño). Aumentos: caja desplazada ±8 % (simula el detector), giros, color, baja resolución, desenfoque, ruido y JPEG. 80 épocas; `best.keras` se elige por IoU de daño en valid.

Escribe en `MyDrive/eggs_v2/runs/v3_dano`. Si esa carpeta ya existe, el script se detiene en lugar de sobrescribirla: en ese caso cambia `RUN` en la celda 1.

In [ ]:
!python {REPO}/dano/train_seg.py {RUN} 80 2>&1 | grep -vE "oneDNN|UserWarning|data_adapter|cuda_|computation_placer|absl|E0000|W0000|I0000"

## 5. Exportación y evaluación
Convierte `best.keras` a `.tflite` en FP32 y FP16 (entrada NHWC `[1,192,192,3]`), comprueba que dan lo mismo que Keras y evalúa la **tubería completa en test**: el detector `v2` encuentra la caja, se recorta como en la app y se segmenta. Guarda todo en `MyDrive/eggs_v2/exports/v3_dano/`, junto con `resumen.json` y `ejemplos_test.jpg`.

In [ ]:
OUT = EXPORTS_DIR / RUN
!python {REPO}/dano/export_eval.py {RUN} {OUT} {DET} 2>&1 | grep -vE "oneDNN|WARNING|absl|E0000|W0000|I0000|cuda_"
display(Image(str(OUT / 'ejemplos_test.jpg'), width=900))
!ls -la {OUT}